In [9]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [11]:

embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    #dimensions=32
)

In [12]:
vectorstore = Chroma(persist_directory="./Astronomy_Cosmology_RAG_Reference",
                               embedding_function=embedding
                               )

In [14]:
retriever = vectorstore.as_retriever(search_type = 'mmr',
                                    search_kwargs = {'k':3, 'lambda_mult':0.7}
                                    )

In [15]:
TEMPLATE = '''
Answer the following question.
{question}

to answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this content is taken from in the format:
Resources: *Subtopic*
where *SubTopic* should be substitute with the SubTopic of all resources.

'''

prompt_template = PromptTemplate.from_template(TEMPLATE)

In [16]:
chat = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    #max_output_tokens=1000,
    seed=42
)

In [27]:
question = "Elaborate difference between Astronoly and Cosmology?"

In [28]:
chain = (
    {'context':retriever,
    'question': RunnablePassthrough()} | prompt_template | chat
)

In [29]:
chain.invoke(question)

AIMessage(content=[{'type': 'text', 'text': "Based on the provided context, the differences between Astronomy and Cosmology are as follows:\n\n* **Astronomy** is the scientific study of celestial objects and phenomena beyond Earth's atmosphere. This includes stars, planets, moons, asteroids, comets, galaxies, black holes, radiation, and the physical processes governing them. It focuses on investigating individual or local systems (such as stars and galaxies).\n\n* **Cosmology** is a specific branch of physics and astronomy concerned with the universe as a whole. It focuses on the universe's origin, large-scale structure, evolution, geometry, composition, and possible long-term future. It primarily studies the global properties of the universe and its evolution on the largest scales.\n\n**Key Distinction:** Astronomy focuses on individual or local celestial systems, whereas cosmology studies the global properties and evolution of the universe as a whole on the largest scales.\n\nResourc